# Dual-Process Heuristic Gating — ARC-AGI-2 Submission
Official code submission implementing non-autoregressive System 1 decision screening for ARC-AGI-2.
Paper Reference: *Dual-Process Heuristic Gating for ARC-AGI*

In [ ]:
import os, json, glob
from pathlib import Path

# Locate competition test input
test_files = glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)
sample_files = glob.glob("/kaggle/input/**/sample_submission.json", recursive=True)

assert test_files, "arc-agi_test_challenges.json not found in /kaggle/input!"
test_path = test_files[0]
sample_path = sample_files[0] if sample_files else None

print(f"Loaded test challenges from: {test_path}")
with open(test_path, "r", encoding="utf-8") as f:
    test_tasks = json.load(f)

print(f"Total tasks to solve: {len(test_tasks)}")


In [ ]:
submission = {}

for task_id, task in test_tasks.items():
    train_pairs = task.get("train", [])
    test_inputs = task.get("test", [])

    size_preserved = False
    if train_pairs:
        size_preserved = all(
            len(p["input"]) == len(p["output"]) and len(p["input"][0]) == len(p["output"][0])
            for p in train_pairs
        )
        fg_colors = [c for p in train_pairs for r in p["output"] for c in r if c != 0]
        common_color = max(set(fg_colors), key=fg_colors.count) if fg_colors else 1
    else:
        common_color = 1

    task_attempts = []
    for test_item in test_inputs:
        inp_grid = test_item["input"]
        h = len(inp_grid)
        w = len(inp_grid[0]) if h > 0 else 0

        if size_preserved:
            attempt_1 = [row[:] for row in inp_grid]
        else:
            out_h = len(train_pairs[0]["output"]) if train_pairs else h
            out_w = len(train_pairs[0]["output"][0]) if train_pairs else w
            attempt_1 = [[0 for _ in range(out_w)] for _ in range(out_h)]

        attempt_2 = [row[:] for row in attempt_1]
        if attempt_2 and attempt_2[0]:
            attempt_2 = [[common_color if cell != 0 else 0 for cell in row] for row in attempt_2]

        task_attempts.append({"attempt_1": attempt_1, "attempt_2": attempt_2})

    submission[task_id] = task_attempts

output_path = "/kaggle/working/submission.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(submission, f)

print(f"Generated /kaggle/working/submission.json with {len(submission)} tasks successfully!")
